In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, joblib
from pathlib import Path

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from scipy.stats import loguniform, randint

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print("Setup OK · Versions :")
import sklearn; print(f"  scikit-learn : {sklearn.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")

Setup OK · Versions :
  scikit-learn : 1.8.0
  pandas       : 3.0.3
  numpy        : 2.4.6


In [2]:
df = pd.read_csv("listings.csv")
print(f"✓ Chargé depuis CSV : {df.shape}")

df.head(3)

✓ Chargé depuis CSV : (84055, 79)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,...,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,3109,https://www.airbnb.com/rooms/3109,20250606142312,2025-06-20,city scrape,zen and calm,Lovely Appartment with one bedroom with a Quee...,Good restaurants<br />very close the Montparna...,https://a0.muscache.com/pictures/miso/Hosting-...,3631,https://www.airbnb.com/users/show/3631,Anne,2008-10-14,"Paris, France",NaN,...,2025-06-03,5.00,5.00,5.00,5.00,5.00,5.00,5.00,7511409139079,f,1,1,0,0,0.08
1,5396,https://www.airbnb.com/rooms/5396,20250606142312,2025-06-19,city scrape,Your perfect Paris studio on Île Saint-Louis,"Cozy, well-appointed and graciously designed s...","You are within walking distance to the Louvre,...",https://a0.muscache.com/pictures/52413/f9bf76f...,7903,https://www.airbnb.com/users/show/7903,Borzou,2009-02-14,"Paris, France",We have spent a lot of time traveling for work...,...,2025-06-05,4.62,4.64,4.59,4.81,4.85,4.95,4.59,7510402838018,f,1,1,0,0,2.32
2,7397,https://www.airbnb.com/rooms/7397,20250606142312,2025-06-20,city scrape,MARAIS - 2ROOMS APT - 2/4 PEOPLE,"VERY CONVENIENT, WITH THE BEST LOCATION !",NaN,https://a0.muscache.com/pictures/67928287/330b...,2626,https://www.airbnb.com/users/show/2626,Franck,2008-08-30,"Paris, France","I am a writer,54, author of novels, books of l...",...,2025-06-03,4.73,4.80,4.45,4.92,4.89,4.94,4.74,7510400829623,f,1,1,0,0,2.20


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84055 entries, 0 to 84054
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            84055 non-null  int64  
 1   listing_url                                   84055 non-null  str    
 2   scrape_id                                     84055 non-null  int64  
 3   last_scraped                                  84055 non-null  str    
 4   source                                        84055 non-null  str    
 5   name                                          84055 non-null  str    
 6   description                                   81177 non-null  str    
 7   neighborhood_overview                         41178 non-null  str    
 8   picture_url                                   84054 non-null  str    
 9   host_id                                       84055 non-null  int64  
 1

In [4]:
print("Shape :", df.shape)
display(df.describe())

Shape : (84055, 79)


,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,...,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,8.405500e+04,8.405500e+04,8.405500e+04,84029.000000,84029.000000,0.0,84055.000000,84055.000000,84055.000000,54297.000000,76918.000000,54007.000000,84055.000000,8.405500e+04,84055.000000,...,84055.000000,84055.000000,5.396300e+04,64498.000000,64486.000000,64489.000000,64479.000000,64488.000000,64480.000000,64479.000000,84055.000000,84055.000000,84055.000000,84055.000000,64498.000000
mean,6.347984e+17,2.025061e+13,1.857889e+08,31.938843,41.159861,NaN,48.864047,2.342917,3.225198,1.211503,1.343457,1.830726,43.545512,5.902691e+02,42.508619,...,5.692975,61.367700,1.817894e+04,4.732790,4.776988,4.667760,4.814846,4.833622,4.823588,4.631523,25.369758,24.193016,1.044031,0.014586,1.045166
std,5.402776e+17,0.000000e+00,2.076564e+08,120.838795,157.249944,NaN,0.018094,0.034203,1.684463,0.555283,0.909540,1.190300,109.597333,3.449318e+04,109.034033,...,11.875125,85.354511,6.508574e+04,0.389992,0.368886,0.438217,0.354538,0.354508,0.303654,0.430997,97.944143,96.282841,10.727246,0.259222,1.350142
min,3.109000e+03,2.025061e+13,2.750000e+02,0.000000,0.000000,NaN,48.815890,2.229896,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000e+00,1.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.150505e+07,2.025061e+13,1.948488e+07,1.000000,1.000000,NaN,48.850758,2.320790,2.000000,1.000000,1.000000,1.000000,2.000000,4.500000e+01,1.000000,...,0.000000,0.000000,0.000000e+00,4.640000,4.710000,4.540000,4.780000,4.810000,4.770000,4.500000,1.000000,1.000000,0.000000,0.000000,0.180000
50%,8.318022e+17,2.025061e+13,7.315135e+07,1.000000,2.000000,NaN,48.865321,2.346677,3.000000,1.000000,1.000000,2.000000,3.000000,3.650000e+02,3.000000,...,0.000000,10.000000,6.240000e+03,4.840000,4.880000,4.800000,4.920000,4.950000,4.920000,4.730000,1.000000,1.000000,0.000000,0.000000,0.560000
75%,1.118822e+18,2.025061e+13,3.389546e+08,4.000000,6.000000,NaN,48.878630,2.367945,4.000000,1.000000,2.000000,2.000000,6.000000,1.125000e+03,6.000000,...,6.000000,104.000000,2.079650e+04,5.000000,5.000000,4.980000,5.000000,5.000000,5.000000,4.890000,3.000000,2.000000,0.000000,0.000000,1.400000
max,1.437099e+18,2.025061e+13,7.007037e+08,7954.000000,8721.000000,NaN,48.901670,2.468360,16.000000,42.000000,41.000000,17.000000,1000.000000,1.000000e+07,1000.000000,...,786.000000,255.000000,3.655680e+06,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,835.000000,835.000000,166.000000,9.000000,67.160000


In [5]:
print(df.isnull().sum())

id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                              0
                                                ...  
calculated_host_listings_count                      0
calculated_host_listings_count_entire_homes         0
calculated_host_listings_count_private_rooms        0
calculated_host_listings_count_shared_rooms         0
reviews_per_month                               19557
Length: 79, dtype: int64


In [9]:
cols_bool = ['host_has_profile_pic', 'host_identity_verified']
for col in cols_bool:
    # Le map transforme 't' en 1 et 'f' en 0. Les valeurs manquantes restent NaN pour le moment.
    df[col] = df[col].map({'t': 1, 'f': 0})
valeurs_uniques = df.nunique().sort_values()[30:]
print(valeurs_uniques)

calculated_host_listings_count_entire_homes       94
estimated_occupancy_l365d                         98
host_acceptance_rate                             101
minimum_nights                                   105
minimum_minimum_nights                           118
maximum_minimum_nights                           137
number_of_reviews_ly                             140
number_of_reviews_ltm                            142
host_listings_count                              147
review_scores_location                           166
review_scores_communication                      168
review_scores_accuracy                           170
review_scores_checkin                            173
review_scores_rating                             181
review_scores_value                              184
host_total_listings_count                        192
availability_eoy                                 199
review_scores_cleanliness                        203
maximum_nights                                

In [12]:
df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
corrs = df.select_dtypes(include=np.number).corr()['price'].sort_values(ascending=False)
top_corr = corrs.head(40)
print(top_corr)

price                                           1.000000
estimated_revenue_l365d                         0.464334
bathrooms                                       0.234650
accommodates                                    0.233318
bedrooms                                        0.205691
beds                                            0.181490
availability_30                                 0.117645
availability_eoy                                0.102265
availability_365                                0.099006
availability_60                                 0.096502
availability_90                                 0.096237
maximum_nights                                  0.074676
host_listings_count                             0.068375
host_total_listings_count                       0.067636
host_id                                         0.040676
id                                              0.036011
review_scores_cleanliness                       0.023383
review_scores_location         

In [13]:
# 1. Suppression des colonnes 100% vides, des textes longs et des risques de fuite
cols_to_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped','source', 'name', 'picture_url', 'host_id', 'host_url', 'host_name', 'neighbourhood_group_cleansed', 'bathrooms', 'beds', 'calendar_updated', 
    'description', 'neighborhood_overview', 'host_about', 'host_thumbnail_url', 'host_picture_url', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    'calendar_last_scraped', 'has_availability', 'reviews_per_month', 'license', 'host_neighbourhood', 'host_location','host_since', 'first_review', 'last_review'
]
# On utilise errors='ignore' au cas où certaines auraient déjà été supprimées
df_airbnb = df.drop(columns=cols_to_drop, errors='ignore')

# 2. Parsing de bathrooms_text
# On utilise une expression régulière pour extraire le premier nombre (entier ou décimal)
df_airbnb['bathrooms'] = df_airbnb['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
# On supprime la colonne texte d'origine pour ne garder que la feature numérique
df_airbnb = df_airbnb.drop(columns=['bathrooms_text'])

# 3. Parsing des amenities (Extraction des 5 features minimum exigées)
# On définit un dictionnaire avec le nom de la future colonne et le mot-clé à chercher
amenities_to_extract = {
    'has_wifi': 'Wifi',
    'has_tv': 'TV',
    'has_kitchen': 'Kitchen',
    'has_ac': 'Air conditioning',
    'has_elevator': 'Elevator'
}

for col_name, keyword in amenities_to_extract.items():
    # str.contains cherche le mot-clé. na=False permet de mettre 0 si la ligne est NaN.
    # astype(int) transforme le True/False en 1/0
    df_airbnb[col_name] = df_airbnb['amenities'].str.contains(keyword, case=False, na=False).astype(int)

# On supprime la colonne JSON brute qui n'est plus utile pour le modèle
df_airbnb = df_airbnb.drop(columns=['amenities'])
print(df_airbnb.shape)
df_airbnb.head(5)


(84055, 53)


,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,latitude,longitude,property_type,room_type,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,bathrooms,has_wifi,has_tv,has_kitchen,has_ac,has_elevator
0,within an hour,100%,100%,f,1.0,1.0,"['email', 'phone']",1.0,1.0,Neighborhood highlights,Observatoire,48.83191,2.31870,Entire rental unit,Entire home/apt,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,0,1,0,0
1,within an hour,100%,95%,f,2.0,4.0,"['email', 'phone']",1.0,1.0,Neighborhood highlights,Hôtel-de-Ville,48.85247,2.35835,Entire rental unit,Entire home/apt,...,4.81,4.85,4.95,4.59,f,1,1,0,0,1.0,1,1,1,0,0
2,within an hour,100%,67%,t,1.0,10.0,"['email', 'phone']",1.0,1.0,NaN,Hôtel-de-Ville,48.85909,2.35315,Entire rental unit,Entire home/apt,...,4.92,4.89,4.94,4.74,f,1,1,0,0,1.0,1,1,1,0,0
3,NaN,NaN,NaN,f,1.0,1.0,['phone'],1.0,1.0,NaN,Opéra,48.87417,2.34245,Entire rental unit,Entire home/apt,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,1,1,0,0
4,NaN,NaN,NaN,f,2.0,4.0,"['email', 'phone']",1.0,1.0,NaN,Louvre,48.86006,2.34863,Entire rental unit,Entire home/apt,...,NaN,NaN,NaN,NaN,f,1,1,0,0,1.0,1,0,1,0,1


In [14]:
df_airbnb.info()

<class 'pandas.DataFrame'>
RangeIndex: 84055 entries, 0 to 84054
Data columns (total 53 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_response_time                            52663 non-null  str    
 1   host_response_rate                            52663 non-null  str    
 2   host_acceptance_rate                          57913 non-null  str    
 3   host_is_superhost                             82227 non-null  str    
 4   host_listings_count                           84029 non-null  float64
 5   host_total_listings_count                     84029 non-null  float64
 6   host_verifications                            84029 non-null  str    
 7   host_has_profile_pic                          84029 non-null  float64
 8   host_identity_verified                        84029 non-null  float64
 9   neighbourhood                                 41178 non-null  str    
 1

In [18]:
corrs = df_airbnb.select_dtypes(include=np.number).corr()['price'].sort_values(ascending=False)
top_corr = corrs
print(top_corr)

price                                           1.000000
estimated_revenue_l365d                         0.464334
bathrooms                                       0.236619
accommodates                                    0.233318
bedrooms                                        0.205691
has_ac                                          0.134455
availability_30                                 0.117645
availability_eoy                                0.102265
availability_365                                0.099006
availability_60                                 0.096502
availability_90                                 0.096237
has_tv                                          0.086983
maximum_nights                                  0.074676
host_listings_count                             0.068375
host_total_listings_count                       0.067636
has_elevator                                    0.039241
review_scores_cleanliness                       0.023383
has_wifi                       

In [17]:
df_airbnb.isnull().sum()

host_response_time                              31392
host_response_rate                              31392
host_acceptance_rate                            26142
host_is_superhost                                1828
host_listings_count                                26
host_total_listings_count                          26
host_verifications                                 26
host_has_profile_pic                               26
host_identity_verified                             26
neighbourhood                                   42877
neighbourhood_cleansed                              0
latitude                                            0
longitude                                           0
property_type                                       0
room_type                                           0
accommodates                                        0
bedrooms                                         7137
price                                           30092
minimum_nights              